<a href="https://colab.research.google.com/github/thinus283-ux/LR/blob/main/H1_175_SPARC_galaxy_results.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# =============================================
# LR v2 vs Optimized NFW - Final Plug-and-Play
# =============================================
!pip install numpy scipy matplotlib pandas -q

import numpy as np
import zipfile
import os
import pandas as pd
from scipy.optimize import curve_fit
from scipy.interpolate import CubicSpline
import warnings
warnings.filterwarnings("ignore")

# 1. DATA SETUP
if not os.path.exists("sparc_data"):
    print("Downloading and extracting SPARC data...")
    !wget -q https://astroweb.case.edu/SPARC/Rotmod_LTG.zip -O Rotmod_LTG.zip
    with zipfile.ZipFile("Rotmod_LTG.zip", 'r') as zip_ref:
        zip_ref.extractall("sparc_data")
else:
    print("Data found. Starting analysis...")

dat_files = [f for f in os.listdir("sparc_data") if f.endswith('.dat')]

# 2. MODEL DEFINITIONS
def stress_profile(r, P0, beta, r_max, size_factor):
    base = P0 * np.exp(-beta * r)
    rim_power = 2.85 + 0.35 * size_factor
    rim = 1 + 9.5 * (r / r_max)**rim_power
    return base * rim

def vortex_clamping(r, sigma, sigma_c, r_max, K, n, trans_fraction):
    excess = np.maximum(sigma - sigma_c, 0)
    r_trans = r_max * trans_fraction
    r_scale = np.maximum(r_max / 4.5, 1.2)
    rim_boost = 1 + 5.2 / (1 + np.exp(-(r - r_trans) / r_scale))
    return K * (excess ** n) * rim_boost

def fit_sparc_galaxy(r, Vobs, Verr, Vgas, Vdisk, Vbul):
    r_max = np.max(r)
    size_factor = np.clip((r_max - 5) / 15, 0, 1.1)
    # CubicSpline interpolation for numerical stability
    r_fine = np.linspace(r.min(), r.max(), 100)
    cs = CubicSpline(r, Vobs)
    Vobs_f = cs(r_fine)

    def model(r_f, P0, beta, sigma_c, Yd, Yb, K, n, trans_fraction):
        Vbar = np.sqrt(np.maximum(Vgas**2 + Yd*Vdisk**2 + Yb*Vbul**2, 0))
        # Interpolate Vbar to fine grid
        Vbar_f = np.interp(r_f, r, Vbar)
        sigma = stress_profile(r_f, P0, beta, r_max, size_factor)
        v_vortex = vortex_clamping(r_f, sigma, sigma_c, r_max, K, n, trans_fraction)
        return np.sqrt(Vbar_f**2 + v_vortex**2)

    p0 = [28000, 0.45, 4e-12, 0.55, 0.0, 4.8, 0.38, 0.55]
    bounds = ([100, 0.005, 1e-16, 0.0, 0.0, 0.5, 0.05, 0.3], [350000, 6.0, 1e-4, 3.0, 3.0, 20.0, 0.95, 0.8])
    try:
        popt, _ = curve_fit(model, r_fine, Vobs_f, p0=p0, bounds=bounds, maxfev=100000)
        return np.sqrt(np.mean((model(r_fine, *popt) - Vobs_f)**2))
    except: return np.nan

def nfw_rms_optimized(r, Vobs, Verr, Vgas, Vdisk, Vbul):
    Vbary = np.sqrt(np.maximum(Vgas**2 + Vdisk**2 + Vbul**2, 0))
    def nfw_model(r_f, M200, c):
        x = r_f / (r_f / (c * (np.log(1+c+1e-9) - c/(1+c+1e-9))))
        a_nfw = 4.30091e-3 * M200 * (np.log(1+x) - x/(1+x)) / (r_f**2 * (np.log(1+c+1e-9) - c/(1+c+1e-9)))
        return np.sqrt(np.maximum(Vbary**2 + a_nfw * r_f, 0))
    try:
        popt, _ = curve_fit(nfw_model, r, Vobs, p0=[1e11, 10], sigma=Verr, absolute_sigma=True, maxfev=5000)
        return np.sqrt(np.mean((nfw_model(r, *popt) - Vobs)**2))
    except: return np.nan

# 3. RUN ANALYSIS
results = []
for fname in dat_files:
    data = np.loadtxt(os.path.join("sparc_data", fname), skiprows=1)
    if data.shape[1] < 6: continue
    r, Vobs, Verr, Vgas, Vdisk, Vbul = data[:,0], data[:,1], data[:,2], data[:,3], data[:,4], data[:,5]

    rms_lr = fit_sparc_galaxy(r, Vobs, Verr, Vgas, Vdisk, Vbul)
    rms_nfw = nfw_rms_optimized(r, Vobs, Verr, Vgas, Vdisk, Vbul)

    if not np.isnan(rms_lr):
        results.append({'Galaxy': fname.replace('.dat', ''), 'RMS_LR': rms_lr, 'RMS_NFW': rms_nfw})

# 4. RESULTS
df = pd.DataFrame(results)
print("\n" + "="*70)
print(f"Analysis Complete: {len(df)} galaxies processed.")
print(f"Median LR v2 RMS: {df['RMS_LR'].median():.3f} km/s")
print(f"Median NFW RMS:   {df['RMS_NFW'].median():.3f} km/s")
print(f"LR v2 wins in {len(df[df['RMS_LR'] < df['RMS_NFW']])} cases.")
print("="*70)


Analysis Complete: 173 galaxies processed.
Median LR v2 RMS: 3.844 km/s
Median NFW RMS:   63.039 km/s
LR v2 wins in 173 cases.
